In [1]:
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src/')
import features.prepare_data as pp
import models.ptype_run_preds as rp
import models.score_classifier as score
import report.ptype_visualize as viz
import pandas as pd
import pickle
import os
import boto3
from datetime import datetime
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import hydra
from omegaconf import DictConfig
from utils.logs import get_logger
import yaml
import data.s3_download_sso as sso_load
import data.clean_ceo_summary as ceo_clean

In [54]:
config_path='../params.yaml'
with open(config_path) as conf_file:
    config = yaml.safe_load(conf_file)
logger = get_logger('DATA_LOAD', log_level=config['base']['log_level'])

In [3]:
@hydra.main(config_path='params.yaml')
def main(config: DictConfig) -> None:
    print(config.data_load.sso_profile_name)

/tmp/ipykernel_723201/4294687443.py:1: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  @hydra.main(config_path='params.yaml')


In [4]:
config['data_condition']['train_filter_features']

{'Use?': 'yes', 'Classes': 'multi'}

In [5]:
sso_load.data_download(config_path)
ceo_summary = ceo_clean.import_ceo_summary(config_path)
ceo_batch_list = ceo_clean.id_ceo_batches(config_path, ceo_summary)

2023-10-17 12:51:24,598 — DATA_DOWNLOAD — INFO — No new data downloaded
2023-10-17 12:51:24,696 — CEO_SUMMARY_LOAD — INFO — Summary data loaded


In [6]:
ceo_batch_list

['v08', 'v18', 'v15', 'v19']

In [7]:
X, y = pp.create_xy(ceo_batch_list, 
                    classes='multi', 
                    drop_feats=False,
                    config_path =config_path, 
                    verbose=False)

2023-10-17 12:51:24,823 — CEO_SUMMARY_LOAD — INFO — Summary data loaded
0.0 plots labeled unknown were dropped from v08.
2023-10-17 12:51:24,918 — CEO_SUMMARY_LOAD — INFO — Summary data loaded


0.0 plots labeled unknown were dropped from v18.
2023-10-17 12:51:25,134 — CEO_SUMMARY_LOAD — INFO — Summary data loaded
0.0 plots labeled unknown were dropped from v15.
2023-10-17 12:51:25,201 — CEO_SUMMARY_LOAD — INFO — Summary data loaded
0.0 plots labeled unknown were dropped from v19.
warning needs to be updated
Plot id 08012 has no cloud free imagery and will be removed.
Plot id 08023 has no cloud free imagery and will be removed.
Plot id 08039 has no cloud free imagery and will be removed.
Plot id 08114 has no cloud free imagery and will be removed.
Plot id 08124 has no cloud free imagery and will be removed.
Plot id 08170 has no cloud free imagery and will be removed.
Plot id 08181 has no cloud free imagery and will be removed.
Plot id 08182 has no cloud free imagery and will be removed.
Plot id 08204 has no cloud free imagery and will be removed.
Plot id 08214 has no cloud free imagery and will be removed.
Plot id 08216 has no cloud free imagery and will be removed.
Plot id 08

100%|██████████| 426/426 [00:07<00:00, 54.14it/s]

Class count {0.0: 50190, 1.0: 33306}


In [9]:
X_train, X_test, y_train, y_test = pp.reshape_training_data(X, y, config_path, config['data_condition']['scale_features'])

In [15]:
X_train.shape

(55860, 94)

In [18]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

In [37]:
import models.train as trn


In [38]:
estimator_name = config['train']['estimator_name']

In [39]:
estimators = trn.get_supported_estimator()

In [40]:
estimator_name not in estimators.keys()

False

In [26]:
estimator = estimators['rfc']

In [28]:
model = estimator()
model.fit(X_train, y_train)

RandomForestClassifier()

In [29]:
f1_score(y_test, model.predict(X_test))

0.9410524201646917

In [42]:
X_test.shape

(27636, 94)

In [44]:
pd.DataFrame(X_test).columns

RangeIndex(start=0, stop=94, step=1)

In [47]:
trn.get_supported_metrics().keys()

dict_keys(['accuracy', 'balanced_accuracy', 'precision', 'recall', 'f1', 'roc_auc'])

In [55]:
metrics = trn.get_supported_metrics()
metric_name = config['train']['tuning_metric']

In [56]:
metric_name in metrics.keys()

True

In [57]:
metric_name

'balanced_accuracy'